# CICIoT2023 — Preprocessing P3 (leakage-free + capped stratified sampling)

**Input:** CSV 39 feature + cột `Label` **chuỗi** 34 nhãn gốc (vd `Merged*.csv`), trong `INPUT_DIR`.
**Output:** toàn bộ file hậu tố **`_p3`** trong `OUTPUT_DIR`.

## Sơ đồ (đã chốt)
```
CSV → [1] CLEAN (NaN/rỗng/Inf/dedup hash)
    → [2] NHÃN: label34 (giữ) + label8 (0–7)
    → [3] GIU NGUYEN 39 feature (khong bien doi, khong drop feature nao)
    → [4] SPLIT 1: stratified label34 → 80% large_train : 20% TEST (test FULL, phân phối tự nhiên)
    → [5] CAP: nhãn ATTACK giữ ngẫu nhiên ≤ 300k trong large_train; BENIGN giữ 100%
    → [6] SPLIT 2: stratified label34 → 80% TRAIN : 20% VAL (val cùng phân phối với train)
    → [7] SCALER: fit MinMax + Standard CHỈ trên TRAIN đã cap → transform val/test
    → [8] Báo cáo + class weights
```

## Output
| File | Vai trò |
|---|---|
| `train/val/test_p3.parquet` | raw sau biến đổi (giữ cả `label34` + `label8`) |
| `train/val/test_{standard,minmax}_p3.parquet` | đã scale; **chỉ 1 cột `label` (8 nhãn 0–7)**, không audit |
| `scaler_{standard,minmax}_p3.pkl` | fit trên train đã cap — dùng chung cho training & extractor v5 trên RPi |
| `feature_columns_p3.json` | 39 cột (tên + thứ tự) — GIU NGUYEN |
| `sampling_report_p3.json` | mỗi nhãn: total/test/large_train/kept/train/val + cap_applied — bằng chứng luận văn |
| `class_weights_p3.json` | trọng số 8 lớp (inverse-frequency) cho `CrossEntropyLoss` |
| `audit_report_p3.json` | số dòng loại ở từng bước, std từng feature trên train |

Kích thước dự kiến trên 22M thật: **train ≈ 5,2M | val ≈ 1,3M | test ≈ 4,4M** (14 nhãn flood bị cap).

Trước khi chạy chỉnh: `INPUT_DIR`, `OUTPUT_DIR` (Kaggle: `/kaggle/input/...`, `/kaggle/working`).
Đã smoke-test: cap đúng nhãn vượt trần, Benign miễn cap, test đúng 20%/nhãn không cap,
3 tập không giao nhau, train≈val về phân phối, scaler mean≈0/std≈1, 39 feature.

In [1]:
"""
CICIoT2023 — Preprocessing P3 (leakage-free + capped stratified sampling)
=========================================================================
Input : CSV 39 feature + cột 'Label' CHUỖI (34 nhãn gốc), đặt trong INPUT_DIR.
Output: file hậu tố "_p3" (không đè output cũ).

Sơ đồ (đã chốt với người dùng):
  [1] CLEAN      : ép numeric, NaN/rỗng/Inf drop, dedup toàn cục (hash 64-bit)
  [2] NHÃN       : chuẩn hóa chuỗi → label34 (GIỮ) + map → label8 (0-7)
  [3] FEATURE    : GIU NGUYEN 39 dac trung (khong bien doi, khong drop feature nao)
  [4] SPLIT 1    : stratified theo label34 → 80% large_train : 20% TEST (test FULL, không đụng)
  [5] CAP        : trong large_train, mỗi nhãn ATTACK giữ ngẫu nhiên tối đa
                   CAP_ATTACK mẫu (seed cố định); BENIGN GIỮ 100%
  [6] SPLIT 2    : large_train đã cap → stratified label34 → 80% TRAIN : 20% VAL
                   (val tự cùng phân phối với train — đúng vai trò chọn model)
  [7] SCALER     : fit MinMax + Standard CHỈ trên TRAIN đã cap; transform val/test
  [8] BÁO CÁO    : sampling_report_p3.json (mỗi nhãn lấy bao nhiêu/tổng),
                   audit_report_p3.json, feature_columns_p3.json,
                   class_weights_p3.json (trọng số 8 lớp cho CrossEntropyLoss)

Ghi chú: label34 được GIỮ trong các file raw (train/val/test_p3.parquet).
Các file scaled (*_minmax/*_standard) CHỈ có 1 cột 'label' = 8 nhãn 0-7 để train.
"""

import os, gc, glob, json, time, pickle, logging
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ─────────────────────────── CONFIG ───────────────────────────
INPUT_DIR    = '/kaggle/input/datasets/rsnngo/34-feature-raw-fable-thang3737'           # thư mục chứa CSV (vd Merged*.csv); Kaggle: '/kaggle/input/...'
OUTPUT_DIR   = '/kaggle/working'
LABEL_COL    = 'Label'        # cột nhãn CHUỖI 34 lớp

CHUNK_SIZE   = 1_000_000
TEST_RATIO   = 0.20           # split 1: large_train 80 : test 20
VAL_RATIO    = 0.20           # split 2: train 80 : val 20 (trên large_train đã cap)
CAP_ATTACK   = 300_000        # trần mỗi nhãn attack trong large_train (Benign KHÔNG cap)
RANDOM_STATE = 42
SUFFIX       = 'p3.39ft'
# ──────────────────────────────────────────────────────────────

COUNT_COLS = ['ack_count', 'syn_count', 'fin_count', 'rst_count']
DROP_COLS = {}  # GIU NGUYEN 39 dac trung: khong drop feature nao
GROUP_TO_ID = {'Benign':0,'DDoS':1,'DoS':2,'BruteForce':3,
               'Spoofing':4,'Recon':5,'Web-Based':6,'Mirai':7}
KEYWORD_MAP = [  # thứ tự quan trọng: 'ddos' trước 'dos'
    ('benign','Benign'), ('ddos','DDoS'), ('dos','DoS'),
    ('bruteforce','BruteForce'), ('brute','BruteForce'), ('dictionary','BruteForce'),
    ('spoof','Spoofing'), ('mitm','Spoofing'),
    ('recon','Recon'), ('vulnerab','Recon'),
    ('xss','Web-Based'), ('sql','Web-Based'), ('command','Web-Based'),
    ('browser','Web-Based'), ('backdoor','Web-Based'), ('malware','Web-Based'),
    ('upload','Web-Based'), ('web','Web-Based'),
    ('mirai','Mirai'),
]
def label_to_group(lbl: str):
    s = str(lbl).strip().lower()
    for kw, g in KEYWORD_MAP:
        if kw in s:
            return g
    return None

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger(__name__)
OUT = Path(OUTPUT_DIR); OUT.mkdir(parents=True, exist_ok=True)

FULL_PARQUET  = OUT / f'full_dataset_{SUFFIX}.parquet'
SPLIT_PATHS   = {'train': OUT / f'train_{SUFFIX}.parquet',
                 'val':   OUT / f'val_{SUFFIX}.parquet',
                 'test':  OUT / f'test_{SUFFIX}.parquet'}
MINMAX_PKL    = OUT / f'scaler_minmax_{SUFFIX}.pkl'
STANDARD_PKL  = OUT / f'scaler_standard_{SUFFIX}.pkl'
REPORT_JSON   = OUT / f'audit_report_{SUFFIX}.json'
SAMPLING_JSON = OUT / f'sampling_report_{SUFFIX}.json'
FEATURES_JSON = OUT / f'feature_columns_{SUFFIX}.json'
WEIGHTS_JSON  = OUT / f'class_weights_{SUFFIX}.json'
LABEL_MAP_JSON = OUT / f'label_mapping_{SUFFIX}.json'
DIST_L34_PNG  = OUT / f'dist_label34_precap_vs_postcap_{SUFFIX}.png'
DIST_L8_PNG   = OUT / f'dist_label8_precap_vs_postcap_{SUFFIX}.png'

L8, L34 = 'label8', 'label37'
AUDIT = {'suffix': SUFFIX, 'cap_attack': CAP_ATTACK, 'drop_columns': DROP_COLS,
         'group_to_id': GROUP_TO_ID, 'rows_read': 0, 'rows_after_nan_inf': 0,
         'rows_dropped_unknown_label': 0, 'rows_dropped_duplicate': 0, 'rows_final': 0}

def _normalize_cols(cols):
    return (pd.Index(cols).str.strip().str.lower()
            .str.replace(r'[\s/\-]+', '_', regex=True))

# ══════════ BƯỚC 1-3: CLEAN + NHÃN + GIU 39 FEATURE → FULL PARQUET ══════════
def _transform_chunk(chunk, seen_hashes):
    chunk.columns = _normalize_cols(chunk.columns)
    label_col = LABEL_COL.strip().lower().replace(' ', '_').replace('-', '_')
    if label_col not in chunk.columns:
        raise KeyError(f"Khong thay cot nhan '{label_col}'. Cot: {list(chunk.columns)}")
    AUDIT['rows_read'] += len(chunk)

    # [2] label34 (chuẩn hóa chuỗi) + label8
    lbl34 = chunk[label_col].astype(str).str.strip().str.upper()
    grp = lbl34.map(lambda s: label_to_group(s))
    bad = grp.isna() | lbl34.isin(['', 'NAN', 'NONE'])
    if bad.any():
        AUDIT['rows_dropped_unknown_label'] += int(bad.sum())
        chunk, lbl34, grp = chunk[~bad], lbl34[~bad], grp[~bad]
    chunk = chunk.drop(columns=[label_col])
    if chunk.empty:
        return chunk

    # [1] clean numeric / Inf / NaN
    feat = list(chunk.columns)
    chunk[feat] = chunk[feat].apply(pd.to_numeric, errors='coerce')
    ok = ~(chunk.isna().any(axis=1) | np.isinf(chunk.to_numpy()).any(axis=1))
    chunk, lbl34, grp = chunk[ok], lbl34[ok], grp[ok]
    AUDIT['rows_after_nan_inf'] += len(chunk)
    if chunk.empty:
        return chunk

    # [3] GIU NGUYEN 39 dac trung — khong bien doi, khong drop feature nao
    for c in chunk.columns:
        chunk[c] = chunk[c].astype('float32')

    chunk[L8]  = grp.map(GROUP_TO_ID).astype('int8').values
    chunk[L34] = lbl34.values

    # dedup toàn cục (hash trên feature + label34)
    h = pd.util.hash_pandas_object(chunk, index=False).to_numpy()
    keep = np.fromiter((x not in seen_hashes for x in h), bool, len(h))
    keep &= ~pd.Series(h).duplicated().to_numpy()
    seen_hashes.update(h[keep].tolist())
    AUDIT['rows_dropped_duplicate'] += int((~keep).sum())
    return chunk.loc[keep]

def build_full_parquet():
    if FULL_PARQUET.exists():
        log.info('✔ Full parquet P3 đã tồn tại, bỏ qua clean.')
        return
    files = sorted(glob.glob(os.path.join(INPUT_DIR, '**/*.csv'), recursive=True))
    files = [f for f in files if 'label34_distribution' not in os.path.basename(f)]
    assert files, f'Khong tim thay CSV trong {INPUT_DIR}'
    log.info('Clean %d file CSV → %s', len(files), FULL_PARQUET.name)
    writer, schema, seen = None, None, set()
    for fp in tqdm(files, desc='CSV files'):
        for chunk in pd.read_csv(fp, chunksize=CHUNK_SIZE, low_memory=False):
            chunk = _transform_chunk(chunk, seen)
            if chunk.empty:
                continue
            t = pa.Table.from_pandas(chunk, preserve_index=False)
            if writer is None:
                schema = t.schema
                writer = pq.ParquetWriter(FULL_PARQUET, schema, compression='snappy')
            writer.write_table(t.cast(schema))
            AUDIT['rows_final'] += len(chunk)
            del chunk, t; gc.collect()
    writer.close(); del seen; gc.collect()
    log.info('✔ Clean xong: %s dòng unique', f"{AUDIT['rows_final']:,}")

# ══════════ BƯỚC 4-6: SPLIT → CAP → SPLIT (index-level, nhẹ RAM) ══════════
def _plot_distributions(precap34, postcap34, precap8, postcap8, id_to_group):
    """Vẽ biểu đồ phân phối mẫu mỗi nhãn TRƯỚC/SAU khi cap.
    - Hình 1: 34 nhãn gốc (label34).
    - Hình 2: 8 nhóm (label8 0-7)."""
    # ---- Hình 1: 34 nhãn ----
    cats = sorted(precap34, key=lambda c: precap34[c], reverse=True)
    pre  = [precap34[c]  for c in cats]
    post = [postcap34[c] for c in cats]
    x = np.arange(len(cats)); w = 0.42
    fig, ax = plt.subplots(figsize=(max(12, len(cats) * 0.5), 7))
    ax.bar(x - w/2, pre,  w, label='Trước cap (large_train)', color='#d97706')
    ax.bar(x + w/2, post, w, label='Sau cap (kept)',          color='#2563eb')
    ax.axhline(CAP_ATTACK, ls='--', lw=1.2, color='red',
               label=f'CAP_ATTACK = {CAP_ATTACK:,}')
    ax.set_yscale('log')
    ax.set_xticks(x); ax.set_xticklabels(cats, rotation=90, fontsize=8)
    ax.set_ylabel('Số mẫu (log scale)')
    ax.set_title('Phân phối mẫu theo 34 nhãn gốc — trước vs sau khi cap')
    ax.legend(); fig.tight_layout()
    fig.savefig(DIST_L34_PNG, dpi=120); plt.close(fig)
    log.info('✔ Biểu đồ phân phối 34 nhãn → %s', DIST_L34_PNG.name)

    # ---- Hình 2: 8 nhóm ----
    gids  = list(range(8))
    names = [f'{g}:{id_to_group[g]}' for g in gids]
    pre8  = [precap8[g]  for g in gids]
    post8 = [postcap8[g] for g in gids]
    x = np.arange(8); w = 0.42
    fig, ax = plt.subplots(figsize=(11, 6))
    b1 = ax.bar(x - w/2, pre8,  w, label='Trước cap', color='#d97706')
    b2 = ax.bar(x + w/2, post8, w, label='Sau cap',   color='#2563eb')
    ax.set_yscale('log')
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=30, ha='right')
    ax.set_ylabel('Số mẫu (log scale)')
    ax.set_title('Phân phối mẫu theo 8 nhóm (label8 0-7) — trước vs sau khi cap')
    for bars in (b1, b2):
        for bar in bars:
            h = bar.get_height()
            ax.annotate(f'{int(h):,}', (bar.get_x() + bar.get_width()/2, h),
                        ha='center', va='bottom', fontsize=7, rotation=0)
    ax.legend(); fig.tight_layout()
    fig.savefig(DIST_L8_PNG, dpi=120); plt.close(fig)
    log.info('✔ Biểu đồ phân phối 8 nhóm → %s', DIST_L8_PNG.name)


def compute_assignment():
    """Trả về assign(int8): 0=train 1=val 2=test -1=bị cap loại; và sampling report.
    Thứ tự: CAP trên toàn bộ 34 nhãn TRƯỚC -> rồi mới split 80/20 (test) -> split 80/20 (val).
    (Cap áp lên toàn dataset nên test cũng theo phân phối đã cap; tập train nhỏ hơn tự nhiên.)"""
    y34 = pq.read_table(FULL_PARQUET, columns=[L34]).to_pandas()[L34].astype('category')
    codes, cats = y34.cat.codes.to_numpy(), list(y34.cat.categories)
    n = len(codes)
    log.info('Tổng %s dòng, %d nhãn gốc. CAP %s trên 34 nhãn TRƯỚC khi split…',
             f'{n:,}', len(cats), f'{CAP_ATTACK:,}')

    idx = np.arange(n)

    # [CAP TRƯỚC] mỗi nhãn attack giữ ngẫu nhiên ≤ CAP_ATTACK trên TOÀN dataset;
    #            nhãn < CAP_ATTACK hoặc Benign -> giữ nguyên.
    rng = np.random.default_rng(RANDOM_STATE)
    kept_parts, report = [], {}
    for ci, cat in enumerate(cats):
        ii = idx[codes == ci]
        grp = label_to_group(cat)
        if grp != 'Benign' and len(ii) > CAP_ATTACK:
            ii_kept = rng.choice(ii, size=CAP_ATTACK, replace=False)
            capped = True
        else:
            ii_kept, capped = ii, False
        kept_parts.append(ii_kept)
        report[cat] = {'group8': grp, 'total_clean': int(len(ii)),
                       'kept_after_cap': int(len(ii_kept)), 'cap_applied': capped}
    kept_idx = np.concatenate(kept_parts)
    kept_idx.sort()
    log.info('Cap %s: toàn bộ %s → %s (loại %s bản gần-trùng flood)',
             f'{CAP_ATTACK:,}', f'{n:,}', f'{len(kept_idx):,}', f'{n-len(kept_idx):,}')

    # ── Phân phối mẫu mỗi nhãn TRƯỚC vs SAU khi cap (trên TOÀN dataset) ──
    kept_codes = codes[kept_idx]
    precap34  = {cat: int((codes      == ci).sum()) for ci, cat in enumerate(cats)}
    postcap34 = {cat: int((kept_codes == ci).sum()) for ci, cat in enumerate(cats)}
    id_to_group = {v: k for k, v in GROUP_TO_ID.items()}
    precap8  = {g: 0 for g in range(8)}
    postcap8 = {g: 0 for g in range(8)}
    for ci, cat in enumerate(cats):
        gid = GROUP_TO_ID[label_to_group(cat)]
        precap8[gid]  += precap34[cat]
        postcap8[gid] += postcap34[cat]
    _plot_distributions(precap34, postcap34, precap8, postcap8, id_to_group)

    # [SPLIT 1] trên phần ĐÃ CAP: 80 large_train / 20 test (stratify label34)
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=TEST_RATIO, random_state=RANDOM_STATE)
    lt_pos, test_pos = next(sss1.split(kept_idx, kept_codes))
    lt_idx, test_idx = kept_idx[lt_pos], kept_idx[test_pos]

    # [SPLIT 2] trên large_train: 80 train / 20 val
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=VAL_RATIO, random_state=RANDOM_STATE)
    tr_pos, va_pos = next(sss2.split(lt_idx, codes[lt_idx]))
    train_idx, val_idx = lt_idx[tr_pos], lt_idx[va_pos]

    assign = np.full(n, -1, dtype=np.int8)
    assign[train_idx], assign[val_idx], assign[test_idx] = 0, 1, 2
    for ci, cat in enumerate(cats):
        report[cat]['train'] = int((codes[train_idx] == ci).sum())
        report[cat]['val']   = int((codes[val_idx]   == ci).sum())
        report[cat]['test']  = int((codes[test_idx]  == ci).sum())
    log.info('→ train %s | val %s | test %s',
             f'{len(train_idx):,}', f'{len(val_idx):,}', f'{len(test_idx):,}')
    with open(SAMPLING_JSON, 'w') as f:
        json.dump({'cap_attack': CAP_ATTACK, 'benign_capped': False,
                   'order': 'cap(34 nhãn, toàn bộ) -> split80/20(test) -> split80/20(val)',
                   'per_label': report}, f, indent=2)
    return assign


def write_splits(assign):
    for p in SPLIT_PATHS.values():
        if p.exists(): p.unlink()
    writers, offset = {}, 0
    pf = pq.ParquetFile(FULL_PARQUET)
    for batch in tqdm(pf.iter_batches(batch_size=CHUNK_SIZE), desc='Writing splits'):
        df = batch.to_pandas()
        a = assign[offset: offset + len(df)]
        for s, name in ((0,'train'), (1,'val'), (2,'test')):
            sub = df[a == s]
            if len(sub) == 0: continue
            t = pa.Table.from_pandas(sub, preserve_index=False)
            if name not in writers:
                writers[name] = pq.ParquetWriter(SPLIT_PATHS[name], t.schema, compression='snappy')
            writers[name].write_table(t)
        offset += len(df)
        del df, batch; gc.collect()
    for w in writers.values(): w.close()
    for name, p in SPLIT_PATHS.items():
        log.info('✔ %s → %s (%.1f MB)', name, p.name, p.stat().st_size/1e6)

# ══════════ BƯỚC 7: SCALER (fit TRÊN TRAIN ĐÃ CAP) ══════════
def _feature_cols():
    return [c for c in pq.read_schema(SPLIT_PATHS['train']).names if c not in (L8, L34)]

def fit_scalers():
    feats = _feature_cols()
    log.info('Số feature: %d', len(feats))
    std = StandardScaler()
    pf = pq.ParquetFile(SPLIT_PATHS['train'])
    for b in tqdm(pf.iter_batches(batch_size=CHUNK_SIZE, columns=feats), desc='Fit Standard'):
        std.partial_fit(b.to_pandas().values.astype('float64'))
    gmin = np.full(len(feats), np.inf); gmax = np.full(len(feats), -np.inf); tot = 0
    pf = pq.ParquetFile(SPLIT_PATHS['train'])
    for b in tqdm(pf.iter_batches(batch_size=CHUNK_SIZE, columns=feats), desc='MinMax pass'):
        X = b.to_pandas().values.astype('float64')
        np.minimum(gmin, X.min(axis=0), out=gmin); np.maximum(gmax, X.max(axis=0), out=gmax)
        tot += len(X)
    rng_ = gmax - gmin; rng_[rng_ == 0] = 1.0
    mm = MinMaxScaler()
    mm.n_features_in_, mm.n_samples_seen_ = len(feats), tot
    mm.data_min_, mm.data_max_, mm.data_range_ = gmin, gmax, rng_
    mm.scale_, mm.min_ = 1.0/rng_, -gmin/rng_
    AUDIT['train_feature_std'] = {c: float(s) for c, s in zip(feats, np.sqrt(std.var_))}
    AUDIT['zero_std_columns_remaining'] = [c for c,s in AUDIT['train_feature_std'].items() if s==0.0]
    if AUDIT['zero_std_columns_remaining']:
        log.warning('⚠ Cột std=0 trên train: %s', AUDIT['zero_std_columns_remaining'])
    with open(MINMAX_PKL,'wb') as f: pickle.dump(mm,f)
    with open(STANDARD_PKL,'wb') as f: pickle.dump(std,f)
    with open(FEATURES_JSON,'w') as f: json.dump(feats,f,indent=2)
    return mm, std, feats

def transform_and_save(scaler, tag, feats):
    for name, src in SPLIT_PATHS.items():
        dst = OUT / f'{name}_{tag}_{SUFFIX}.parquet'
        if dst.exists(): dst.unlink()
        writer = None
        pf = pq.ParquetFile(src)
        for b in tqdm(pf.iter_batches(batch_size=CHUNK_SIZE), desc=f'{name} [{tag}]'):
            df = b.to_pandas()
            out = pd.DataFrame(scaler.transform(df[feats].values.astype('float32')).astype('float32'),
                               columns=feats)
            lbl8 = df[L8].values.astype('int8')      # NHÃN 8 lop 0-7 cho huan luyen
            assert lbl8.min() >= 0 and lbl8.max() <= 7, \
                f'label ngoai khoang 0-7: [{lbl8.min()},{lbl8.max()}]'
            out['label'] = lbl8                # 1 cot nhan duy nhat, khong audit label34
            t = pa.Table.from_pandas(out, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(dst, t.schema, compression='snappy')
            writer.write_table(t)
            del df, out, b, t; gc.collect()
        writer.close()
        log.info('✔ %s [%s] → %s (%.1f MB)', name, tag, dst.name, dst.stat().st_size/1e6)

def save_label_mapping():
    """Lưu mapping nhãn: label34 (chuỗi) → group8 (chuỗi) → label8 (0-7).
    label8 chính là NHÃN 0-7 dùng để huấn luyện; đã được mã hóa ở bước [2]
    trong _transform_chunk qua GROUP_TO_ID và ghi sẵn vào các file parquet."""
    y34 = pq.read_table(FULL_PARQUET, columns=[L34]).to_pandas()[L34]
    cats = sorted(y34.unique().tolist())
    label34_to_id = {}
    per_label = {}
    for cat in cats:
        grp = label_to_group(cat)
        gid = GROUP_TO_ID[grp]
        label34_to_id[cat] = gid
        per_label[cat] = {'group8': grp, 'label8': gid}
    mapping = {
        'group_to_id': GROUP_TO_ID,
        'id_to_group': {v: k for k, v in GROUP_TO_ID.items()},
        'label34_to_label8': label34_to_id,
        'per_label34': per_label,
        'note': ("label8 la nhan 0-7 dung cho CrossEntropyLoss; "
                 "da map hoa san trong cot 'label8' cua moi file parquet."),
    }
    with open(LABEL_MAP_JSON, 'w') as f:
        json.dump(mapping, f, indent=2, ensure_ascii=False)
    log.info('✔ label mapping (34→8, 0-7) → %s', LABEL_MAP_JSON.name)


def save_class_weights():
    y = pq.read_table(SPLIT_PATHS['train'], columns=[L8]).to_pandas()[L8].values
    cnt = np.bincount(y, minlength=8).astype('float64')
    w = cnt.sum() / (8 * np.maximum(cnt, 1))          # inverse-frequency (sklearn 'balanced')
    with open(WEIGHTS_JSON, 'w') as f:
        json.dump({'train_counts_label8': cnt.astype(int).tolist(),
                   'weights_balanced': w.round(6).tolist(),
                   'usage': "CrossEntropyLoss(weight=torch.tensor(weights_balanced))"}, f, indent=2)
    log.info('✔ class weights (8 lớp) → %s | counts=%s', WEIGHTS_JSON.name, cnt.astype(int).tolist())

def main():
    log.info('═'*62); log.info(' P3: clean → label34/8 → split 80/20 → cap %s → split 80/20', f'{CAP_ATTACK:,}'); log.info('═'*62)
    build_full_parquet()
    assign = compute_assignment()
    write_splits(assign); del assign; gc.collect()
    mm, std, feats = fit_scalers()
    transform_and_save(mm,  'minmax',   feats)
    transform_and_save(std, 'standard', feats)
    save_class_weights()
    save_label_mapping()
    with open(REPORT_JSON, 'w') as f:
        json.dump(AUDIT, f, indent=2, ensure_ascii=False)
    log.info('═'*62); log.info(' HOÀN THÀNH — output tại %s:', OUT.resolve())
    for p in sorted(OUT.iterdir()):
        log.info('   %-42s %9.1f MB', p.name, p.stat().st_size/1e6)

if __name__ == '__main__':
    main()


02:54:04 [INFO] ══════════════════════════════════════════════════════════════
02:54:04 [INFO]  P3: clean → label34/8 → split 80/20 → cap 300,000 → split 80/20
02:54:04 [INFO] ══════════════════════════════════════════════════════════════
02:54:04 [INFO] Clean 63 file CSV → full_dataset_p3.39ft.parquet
CSV files: 100%|██████████| 63/63 [10:05<00:00,  9.60s/it]
03:04:13 [INFO] ✔ Clean xong: 20,701,052 dòng unique
03:04:16 [INFO] Tổng 20,701,052 dòng, 34 nhãn gốc. CAP 300,000 trên 34 nhãn TRƯỚC khi split…
03:04:18 [INFO] Cap 300,000: toàn bộ 20,701,052 → 6,981,789 (loại 13,719,263 bản gần-trùng flood)
03:04:19 [INFO] ✔ Biểu đồ phân phối 34 nhãn → dist_label34_precap_vs_postcap_p3.39ft.png
03:04:20 [INFO] ✔ Biểu đồ phân phối 8 nhóm → dist_label8_precap_vs_postcap_p3.39ft.png
03:04:30 [INFO] → train 4,468,344 | val 1,117,087 | test 1,396,358
Writing splits: 21it [00:22,  1.06s/it]
03:04:53 [INFO] ✔ train → train_p3.39ft.parquet (175.3 MB)
03:04:53 [INFO] ✔ val → val_p3.39ft.parquet (47.0 M